In [ ]:
%matplotlib inline

import subprocess
import sys

import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, "scripts")
import plot_dispersion as pdisp

sim = widgets.IntText(9999, description="Simulation ID")
mph = widgets.FloatText(171, description="Ball Speed (mph)")
deg = widgets.FloatText(10, description="Launch Angle")
back = widgets.FloatText(2545, description="Backspin")
side = widgets.FloatText(10, description="Sidespin")
tgt = widgets.FloatText(282, description="Target Carry")
rad = widgets.FloatText(15, description="Target Radius")
seed = widgets.Text("", description="Seed (Optional)")
go = widgets.Button(description="Run")


def run(_):
    sid = int(sim.value)
    cmd = [
        str(_TEST / "golf_sim"),
        str(sid),
        str(mph.value),
        str(deg.value),
        str(back.value),
        str(side.value),
        str(tgt.value),
        str(rad.value),
    ]
    if seed.value.strip():
        cmd.append(seed.value.strip())
    subprocess.run(["make", "-C", str(_TEST)], check=True)
    subprocess.run(cmd, check=True)

    x, z = pdisp.load_landings_csv(f"output/csv/results_sim{sid}.csv")
    td, tr = pdisp.load_run_metadata(sid)
    pad_x = max(15.0, 0.08 * (float(x.max()) - float(x.min()) + 1.0))
    pad_z = max(20.0, 0.06 * (float(z.max()) - float(z.min()) + 1.0))
    lims = pdisp._axis_limits(x, z, td, tr, pad_x, pad_z)
    x0, x1, z0, z1 = lims

    fig, ax = plt.subplots(figsize=(9, 7))
    fig.patch.set_facecolor("#1e2a1f")
    pdisp.draw_fairway_scene(ax, x0, x1, z0, z1, td)
    n, n_in, mean = pdisp.plot_shots(ax, x, z, td, tr)
    ax.set_xlim(x0, x1)
    ax.set_ylim(z0, z1)
    ax.set_aspect("equal", adjustable="box")
    pct = 100 * n_in / n if n else 0.0
    ax.set_title(
        f"{n:,} shots · {pct:.1f}% in target · avg carry {mean:.0f} yd",
        color="white",
    )
    ax.tick_params(colors="white")
    ax.set_xlabel("Lateral (yds)", color="white")
    ax.set_ylabel("Distance (yds)", color="white")
    plt.show()


go.on_click(run)
display(widgets.VBox([sim, mph, deg, back, side, tgt, rad, seed, go]))